In [2]:
import sys
import glob
import torch

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats

from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path


sys.path.append('../utils/')
sys.path.append('../src/model/')

sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utils.dprime import recompute_dprime_by_isi_per_subject
from utils.reliability import compute_itemwise_split_half_reliability

from utils.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utils.loading import load_results_with_exclusion_2
from utils.dprime import recompute_dprime_by_isi_per_subject

from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utils.reliability import compute_power_curve
from utils.plotting import plot_power_curve

import DistanceMemoryModel
import encoders


import json
import glob
import sys
import os

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import norm
from collections import defaultdict

from IPython.display import clear_output

import argparse

# noise_vars = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
# criterions = [80.0, 140.0, 4.0, 40.0, 20.0, 10.0, 30.0, 80.0, 140.0, 4.0, 40.0, 20.0, 10.0, 30.0]

# noise_vars = [1.0,  1.0,  1.0,  1.0,  1.0,  1.0,   1.0,  0.5,  0.5,  0.5,  0.5,  0.5,  0.5,  0.5]
# criterions = [42.0, 48.0, 52.0, 58.0, 62.0, 68.0, 72.0, 42.0, 48.0, 52.0, 58.0, 62.0, 68.0, 72.0]


#criterions = [61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0] 
criterions = [30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0]
noise_vars = [0.1] * len(criterions)



param_idx = 3
pc_dims=256


best_noise_var = noise_vars[param_idx]
best_criterion = criterions[param_idx]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"

base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"

seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}

hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)



safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

# where you want the results
CSV_PATH = f"/om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel_PCs{pc_dims}/{safe_name}/"

ensure_dir(save_dir)
ensure_dir(CSV_PATH)
print(save_dir)


CSV_PATH = f"/om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel_PCs{pc_dims}/{safe_name}/isi16_runs.csv"
    

memory_model = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=texture_model,
    noise_variance=best_noise_var,
    criterion=best_criterion,
    device='cuda'
)

cuda
2008997
Directory already exists: /om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only/atexts-len120
Directory already exists: /om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel_PCs256/atexts-len120/
/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only/atexts-len120


In [4]:
texture_model(texture_list[0])

tensor([ 9.4501e+00,  2.9881e+00,  3.1584e-01, -5.0570e+00, -4.6874e+00,
         9.3479e+00, -6.9374e-01,  6.9270e-01,  1.8929e+00,  5.3083e+00,
         1.2124e+00,  5.7160e+00,  3.2079e+00,  1.4932e-02, -4.0543e+00,
         2.4271e+00, -3.5085e-01,  7.5115e+00,  3.4132e+00,  6.1459e-01,
        -1.5528e+00,  2.0770e+00, -3.1709e+00, -2.3054e-01, -6.3680e-01,
        -3.1129e+00,  1.7238e+00,  1.8955e+00,  1.5815e+00,  1.0495e+00,
        -6.5779e-01, -1.7487e+00,  1.4278e+00,  2.9442e+00,  5.0506e-01,
        -5.2474e+00, -2.3808e-01,  1.3726e+00,  2.2363e-03, -1.0982e+00,
        -1.8977e+00,  6.4048e+00,  2.2122e+00,  4.8391e-01, -4.8467e-01,
         1.1108e+00,  6.1823e-01,  3.1538e-01, -2.6248e-01, -6.5435e-01,
        -3.9004e+00,  1.6681e+00, -6.6347e-01, -1.5401e+00,  4.5063e+00,
         1.7170e+00,  2.1035e+00,  1.3400e+00,  6.3891e-01,  5.1752e-01,
         1.1651e+00,  4.7092e+00,  5.7873e+00,  1.5544e+00, -1.5351e+00,
        -2.2357e-01, -5.7711e+00, -1.8394e+00, -2.3

In [3]:
statistics_set

<module 'texture_prior.params.statistics_set' from '/om2/user/jmhicks/projects/TextureStreaming/code/texture_prior/params/statistics_set.py'>